# Bank Ticket Priority Router final evaluation
Upload the full repository ZIP. Offline verification reproduces submitted results without a key. Optional fresh model inference consumes credit and writes separately under reruns/.


In [ ]:
from google.colab import files
import os, pathlib, zipfile, sys, subprocess
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload one complete repository ZIP.')
destination = pathlib.Path('/content/bank_ticket_submission')
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for name in archive.namelist():
        parts = pathlib.PurePosixPath(name)
        if parts.is_absolute() or '..' in parts.parts or '\\' in name:
            raise ValueError('Archive has an unsafe path.')
    archive.extractall(destination)
candidates = [p.parent for p in destination.rglob('app.py')
              if (p.parent/'ticket_router.py').is_file() and (p.parent/'data/test.csv').is_file()]
if len(candidates) != 1:
    raise ValueError('Expected one project with app.py, ticket_router.py and data/test.csv. Use a fresh runtime if needed.')
os.chdir(candidates[0])
sys.path.insert(0, str(pathlib.Path.cwd()))
print('Repository loaded:', pathlib.Path.cwd().name)


## Offline submitted-result verification
No dependency installation and no API request.


In [ ]:
subprocess.run([sys.executable, 'eval_saved.py'], check=True)
subprocess.run([sys.executable, 'reproduce_selection.py'], check=True)


## Fresh baseline run
These real offline predictions are stored under reruns/final_200/.


In [ ]:
subprocess.run([sys.executable, 'run_final.py', 'baselines'], check=True)


## Optional fresh LLM evaluation
Only run the following cells if you want another real 200-ticket API run. The submitted results remain unchanged. Interrupted calls resume from reruns/final_200/ when settings match.


In [ ]:
from getpass import getpass
key = getpass('OpenRouter key: ').strip()
if not key or not key.isascii() or any(ord(c) <= 32 or ord(c) == 127 for c in key):
    raise ValueError('Enter only a valid ASCII key without spaces.')
os.environ['OPENROUTER_API_KEY'] = key
del key


In [ ]:
subprocess.run([sys.executable, 'run_final.py', 'llm'], check=True)


## Download new outputs
This exports only reruns, never environment credentials. Run before closing the runtime.


In [ ]:
import shutil
shutil.make_archive('/content/Ticket_Router_New_Runs', 'zip', root_dir='reruns')
files.download('/content/Ticket_Router_New_Runs.zip')
